In [1]:
# Importando as bibliotecas
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

import matplotlib.pyplot as plt
import seaborn as sns
from features.feature_extractor_v2 import extract_features
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


In [2]:
from features.feature_extractor_v2 import extract_features

print(
    extract_features(
        "https://google.login-security.xyz"
    ).keys()
)

dict_keys(['NumDots', 'SubdomainLevel', 'PathLevel', 'UrlLength', 'NumDash', 'NumDashInHostname', 'AtSymbol', 'TildeSymbol', 'NumUnderscore', 'NumPercent', 'NumQueryComponents', 'NumAmpersand', 'NumHash', 'NumNumericChars', 'PctNumericChars', 'NoHttps', 'IpAddress', 'DomainInSubdomains', 'DomainInPaths', 'HostnameLength', 'PathLength', 'QueryLength', 'HttpsInHostname', 'HostnameHasDigit', 'HostnameDigitCount', 'NumSensitiveWords', 'DoubleSlashInPath', 'Has_login', 'Has_verify', 'Has_secure', 'Has_account', 'Has_update', 'Has_confirm', 'Has_signin', 'Has_bank', 'Has_payment', 'Has_wallet', 'Has_password', 'Has_security', 'Has_support', 'SuspiciousTLD', 'ManySubdomains', 'KnownBrandInURL', 'BrandInSubdomain', 'BrandInPath', 'BrandMismatch', 'SpecialCharRatio', 'LongURL', 'VeryLongURL', 'UrlEntropy', 'FreeHosting', 'RandomLookingHostname'])


In [3]:
import features.feature_extractor_v2 as fe

print(fe.__file__)

c:\Users\u513127\Downloads\phishing-project-kaggle-main\phishing-project-kaggle-main\ml\features\feature_extractor_v2.py


In [4]:
df_balanced = pd.read_csv(
    "../datasets/raw/balanced_urls.csv"
)

In [5]:
df_balanced.head()

,url,label,result
0,https://www.google.com,benign,0
1,https://www.youtube.com,benign,0
2,https://www.facebook.com,benign,0
3,https://www.baidu.com,benign,0
4,https://www.wikipedia.org,benign,0


In [6]:
df_balanced["result"].value_counts()

result
0    316254
1    316254
Name: count, dtype: int64

In [7]:
features = df_balanced["url"].apply(
    lambda url: pd.Series(
        extract_features(url)
    )
)

In [8]:
df_features = pd.concat(
    [
        df_balanced,
        features
    ],
    axis=1
)

In [9]:
df_features.to_csv(
    "../datasets/processed/df_balanced_v2.csv",
    index=False
)

In [10]:
print(df_features.shape)
df_features.head()

(632508, 55)


,url,label,result,NumDots,SubdomainLevel,PathLevel,UrlLength,NumDash,NumDashInHostname,AtSymbol,...,KnownBrandInURL,BrandInSubdomain,BrandInPath,BrandMismatch,SpecialCharRatio,LongURL,VeryLongURL,UrlEntropy,FreeHosting,RandomLookingHostname
0,https://www.google.com,benign,0,2.0,1.0,0.0,22.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.663533,0.0,0.0
1,https://www.youtube.com,benign,0,2.0,1.0,0.0,23.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.762267,0.0,0.0
2,https://www.facebook.com,benign,0,2.0,1.0,0.0,24.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.855389,0.0,0.0
3,https://www.baidu.com,benign,0,2.0,1.0,0.0,21.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.880180,0.0,0.0
4,https://www.wikipedia.org,benign,0,2.0,1.0,0.0,25.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.813661,0.0,0.0


In [11]:
from sklearn.model_selection import train_test_split

df_train, df_holdout = train_test_split(
    df_features,
    test_size=0.20,
    stratify=df_features["result"],
    random_state=42
)

In [12]:
print(df_train.shape)
print(df_holdout.shape)

(506006, 55)
(126502, 55)


In [13]:
X_train = df_train.drop(
    columns=[
        "url",
        "label",
        "result"
    ],
    errors="ignore"
)

# remover feature enviesada
X_train = X_train.drop(
    columns=["NoHttps"],
    errors="ignore"
)

y_train = df_train["result"]

In [14]:
X_test = df_holdout.drop(
    columns=[
        "url",
        "label",
        "result"
    ],
    errors="ignore"
)

# remover feature enviesada
X_test = X_test.drop(
    columns=["NoHttps"],
    errors="ignore"
)

y_test = df_holdout["result"]

In [15]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(506006, 51)
(126502, 51)
(506006,)
(126502,)


In [16]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 506006 entries, 318973 to 498253
Data columns (total 51 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   NumDots                506006 non-null  float64
 1   SubdomainLevel         506006 non-null  float64
 2   PathLevel              506006 non-null  float64
 3   UrlLength              506006 non-null  float64
 4   NumDash                506006 non-null  float64
 5   NumDashInHostname      506006 non-null  float64
 6   AtSymbol               506006 non-null  float64
 7   TildeSymbol            506006 non-null  float64
 8   NumUnderscore          506006 non-null  float64
 9   NumPercent             506006 non-null  float64
 10  NumQueryComponents     506006 non-null  float64
 11  NumAmpersand           506006 non-null  float64
 12  NumHash                506006 non-null  float64
 13  NumNumericChars        506006 non-null  float64
 14  PctNumericChars        506006 non-nu

In [17]:
rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

In [18]:
rf.fit(
    X_train,
    y_train
)

,n_estimators,50
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
rf_pred = rf.predict(X_test)

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print(
    "Accuracy:",
    accuracy_score(y_test, rf_pred)
)

print(
    "Precision:",
    precision_score(y_test, rf_pred)
)

print(
    "Recall:",
    recall_score(y_test, rf_pred)
)

print(
    "F1:",
    f1_score(y_test, rf_pred)
)

Accuracy: 0.9523248644290209
Precision: 0.9681568268097919
Recall: 0.935416040853109
F1: 0.9515048688114632


In [21]:
print(
    classification_report(
        y_test,
        rf_pred
    )
)

              precision    recall  f1-score   support

           0       0.94      0.97      0.95     63251
           1       0.97      0.94      0.95     63251

    accuracy                           0.95    126502
   macro avg       0.95      0.95      0.95    126502
weighted avg       0.95      0.95      0.95    126502



In [22]:
confusion_matrix(
    y_test,
    rf_pred
)

array([[61305,  1946],
       [ 4085, 59166]])

In [23]:
joblib.dump(
    rf,
    "../models/scenario_3c1/rf_scenario_3c1.pkl"
)

['../models/scenario_3c1/rf_scenario_3c1.pkl']

In [24]:
joblib.dump(
    list(X_train.columns),
    "../artifacts/scenario_3c1/features_scenario_3c1.pkl"
)

['../artifacts/scenario_3c1/features_scenario_3c1.pkl']

In [25]:
feature_columns = list(X_train.columns)

joblib.dump(
    feature_columns,
    "../artifacts/scenario_3c1/feature_columns.pkl"
)

['../artifacts/scenario_3c1/feature_columns.pkl']

In [26]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
1,SubdomainLevel,0.199358
18,HostnameLength,0.092046
48,UrlEntropy,0.066435
0,NumDots,0.059532
19,PathLength,0.056564
3,UrlLength,0.051985
16,DomainInSubdomains,0.042178
14,PctNumericChars,0.038695
45,SpecialCharRatio,0.038044
43,BrandInPath,0.034827


In [27]:
df_features["complexity"] = (
    df_features["SubdomainLevel"]
    + df_features["PathLevel"]
)

df_features.sort_values(
    "complexity"
)[
    ["url", "result"]
].head(20)

,url,result
434267,pinuppopup.com,1
434268,shaparaknet.ir,1
434298,sgn80.com,1
434316,semoon.mn,1
434328,384242.799866074.cn,1
434329,amsonsmanpower.com,1
434331,mining24.info,1
434303,yangzirivercorp.com.au,1
434214,andromedatechnologies.co.in,1
434215,sign-ln-lcloud.com,1


In [28]:
train_pred = rf.predict(X_train)

print(
    classification_report(
        y_train,
        train_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    253003
           1       1.00      1.00      1.00    253003

    accuracy                           1.00    506006
   macro avg       1.00      1.00      1.00    506006
weighted avg       1.00      1.00      1.00    506006



In [29]:
pd.crosstab(
    df_features["label"],
    df_features["result"]
)

result,0,1
label,,
benign,316254,0
malicious,0,316254


In [32]:
benign = df_features[df_features["result"] == 0]

benign["url"].sample(100, random_state=42).tolist()

['https://www.academon.com/Essay-Transcontinental-Railroad/29187',
 'https://www.picasaweb.google.com/TDSSkupina',
 'https://www.picktorrent.com/download/dc/3766609/bruning-saviours-discography/',
 'https://www.cbc.ca/news/quebecvotes2008/ridings/053/',
 'https://www.ccspringvalley.org/',
 'https://www.brooklynvegan.com/archives/2011/01/hunx_his_punx_t.html',
 'https://www.profile.ultimate-guitar.com/starleigh/blog/30351/',
 'https://www.wibw.com/sports/headlines/Kansas_Football_Set_To_Open_2011_Season_Hosting_McNeese_State_Saturday_128624373.html',
 'https://www.rhodeislandentertainmentlawyers.com/',
 'https://www.houseofnames.com/fairholm-family-crest',
 'https://www.youtube.com/watch?v=pKWMSUw4Eo8',
 'https://www.musicstack.com/records-cds/ras+kass',
 'https://www.youtube.com/watch?v=rIh6wYHHyjE',
 'https://www.ontheradio.net/kllc',
 'https://www.en.wikipedia.org/wiki/Fran%C3%A7ois_Benjamin',
 'https://www.weather.com/outlook/driving/interstate/cities?ix=80&reg=us&dir=we',
 'https:/